# LushProtein — EDA from Raw Data (Data Quality & Finals Filters)

**Fully self-contained.** Loads the five raw data folders and reproduces mid-term EDA plus
post-midterm **finals cohort filters** (DQ-02/03/04, LP-F01–F04, Layer 0/3).

Includes filter funnel metrics and **20+ visualizations** documenting data quality decisions.


In [1]:
# ── 0. Install dependencies (run this cell first) ───────────────────────────
import importlib.util
import subprocess
import sys

REQUIRED = [
    "pandas", "numpy", "matplotlib", "seaborn", "scikit-learn",
    "openpyxl", "pyarrow",
]

missing = [p for p in REQUIRED if importlib.util.find_spec(p) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All dependencies already installed.")


Installing: scikit-learn


In [2]:
import warnings
warnings.filterwarnings("ignore")

import json
import os
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
try:
    from IPython.display import display
except ImportError:
    display = print
import matplotlib.ticker as mticker

# ── Project paths ─────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "1.customer_transaction").exists():
    alt = PROJECT_ROOT.parent
    if (alt / "1.customer_transaction").exists():
        PROJECT_ROOT = alt
    else:
        raise FileNotFoundError(
            "Open this notebook from the project root (folder containing 1.customer_transaction/)."
        )
os.chdir(PROJECT_ROOT)

RAW_ORDERS_DIR = PROJECT_ROOT / "1.customer_transaction"
STANDALONE_OUT = PROJECT_ROOT / "standalone_outputs"
STANDALONE_OUT.mkdir(exist_ok=True)

def _glob_one(folder: Path, pattern: str) -> Path:
    matches = sorted(folder.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No file matching {pattern} in {folder}")
    return matches[0]

ORDER_FILES = sorted(RAW_ORDERS_DIR.glob("1_*.xlsx"))
PRODUCTS_FILE = _glob_one(PROJECT_ROOT / "2.product_master", "*.xlsx")
DISCOUNTS_FILE = _glob_one(PROJECT_ROOT / "3.Discounts", "*.csv")
CAMPAIGNS_FILE = _glob_one(PROJECT_ROOT / "4.Campaigns", "*.csv")
RECHARGE_DIR = PROJECT_ROOT / "5.Recharge_data"

EXCLUDE_HANDLE = "better-whey-protein-elite"
EXCLUDE_MONTHS = {7, 11}
ANALYSIS_START = pd.Timestamp("2022-01-01", tz="Asia/Singapore")
ANALYSIS_DATE = pd.Timestamp("2026-04-30", tz="UTC")
MARGIN_PROXY = 0.40
N_TIERS = 5
DECILE_CHART_ORDER = [f"D{i}" for i in range(10, 0, -1)]

FX_RATES_TO_SGD = {"SG": 1.0, "MY": 1.0 / 3.30, "HK": 1.0 / 6.10}
PRODUCT_MAP = {
    "lean-protein": "Lean Protein", "lean_protein": "Lean Protein",
    "clear-protein": "Clear Protein", "clear_protein": "Clear Protein",
    "collagen": "Collagen Glow", "soy-protein": "Soy Protein",
    "protein-bar": "Protein Bar", "shaker": "Accessories", "starter-kit": "Accessories",
}
MARKETPLACE_KEYWORDS = ["shopee", "lazada", "tokopedia", "redmart", "grab"]
CATEGORIES = [
    "Clear Protein", "Lean Protein", "Collagen Glow",
    "Accessories", "Soy Protein", "Other", "Unknown",
]

print("Project root:", PROJECT_ROOT)
print("Order files:", len(ORDER_FILES))
print("Products:", PRODUCTS_FILE.name)
print("Output dir:", STANDALONE_OUT)



Project root: c:\Users\adity\Documents\Aditya SMU\SMU Sem 5\Lush Protein SMU X\LushProtein_Project_Data_20260505
Order files: 7
Products: 2_1.products_master_20260505.xlsx
Output dir: c:\Users\adity\Documents\Aditya SMU\SMU Sem 5\Lush Protein SMU X\LushProtein_Project_Data_20260505\standalone_outputs


In [3]:
# ── Helper functions (mirrors EDA/00_config.py + aditya_findings/_shared.py) ─

def classify_product(handle) -> str:
    if pd.isna(handle):
        return "Unknown"
    h = str(handle).lower()
    for kw, label in PRODUCT_MAP.items():
        if kw in h:
            return label
    return "Other"


def classify_channel(row) -> str:
    tags = str(row.get("Tags", "") or "").lower()
    utm = str(row.get("Browser: UTM Source", "") or "").lower()
    name = str(row.get("Name", "") or "").lower()
    if any(k in tags for k in MARKETPLACE_KEYWORDS):
        return "Marketplace"
    if "subscription" in tags or "yotpo subscriptions" in tags or "lpsg" in name[:4]:
        return "Subscription"
    if utm in ("facebook", "instagram", "tiktok"):
        return "Paid Social"
    if utm in ("google", "bing"):
        return "Paid Search"
    if utm == "affiliate":
        return "Affiliate"
    if utm in ("shopify_email", "email", "klaviyo"):
        return "Email"
    return "Direct / Organic"


def store_prefix(name: str) -> str:
    if pd.isna(name):
        return "Unknown"
    n = str(name).upper().replace("#", "")
    if n.startswith("LPMY"):
        return "MY"
    if n.startswith("LPHK"):
        return "HK"
    if n.startswith("LPSG") or n.startswith("LP"):
        return "SG"
    return "Other"


def assign_decile(series: pd.Series) -> pd.Series:
    ranks = series.rank(method="first", ascending=True)
    return pd.qcut(ranks, q=10, labels=DECILE_CHART_ORDER)


def crm_tier(row) -> str:
    if row.get("is_top_both"):
        return "VIP"
    if row.get("is_top_profit") or row.get("profit_decile_true") == "D1":
        return "Profit_D1"
    if row.get("is_top_freq") or row.get("freq_decile_true") == "D1":
        return "Freq_D1"
    return "Standard"


def _safe_str(val, default="") -> str:
    """Convert cell values to str without boolean checks on pd.NA."""
    if val is None:
        return default
    try:
        if pd.isna(val):
            return default
    except (TypeError, ValueError):
        pass
    s = str(val).strip()
    if s in ("nan", "None", "<NA>", ""):
        return default
    return s


def sku_label(row) -> str:
    handle = _safe_str(row.get("Line: Product Handle"), "unknown")
    if handle == "unknown":
        handle = _safe_str(row.get("Line: SKU"), "unknown")
    variant = _safe_str(row.get("Line: Variant Title"), "")
    if "/" in variant:
        flavour = variant.split("/")[-1].strip()
    else:
        flavour = variant[:30] if variant else "default"
    return f"{handle}|{flavour}"[:80]


def clean_object_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.select_dtypes(include="object").columns:
        out[col] = out[col].where(out[col].notna(), other=pd.NA)
        out[col] = out[col].apply(lambda x: str(x) if pd.notna(x) else pd.NA)
    return out


def build_cogs_map(products_df: pd.DataFrame) -> dict[str, float]:
    """COGS from product master Cost per item (Variant SKU key)."""
    cost_map: dict[str, float] = {}
    sku_col = "Variant SKU" if "Variant SKU" in products_df.columns else "SKU"
    cost_col = "Cost per item" if "Cost per item" in products_df.columns else None
    if cost_col is None:
        return cost_map
    for _, r in products_df[[sku_col, cost_col]].dropna(subset=[cost_col]).iterrows():
        cost_map[str(r[sku_col]).strip()] = float(r[cost_col])
    return cost_map


print("Helpers loaded.")

def assign_decile_5(series: pd.Series, n_tiers: int = N_TIERS) -> pd.Series:
    """D1 = best. Five tiers for Solution 1 (founder request)."""
    ranks = series.rank(method="first", ascending=True)
    return pd.qcut(ranks, q=n_tiers, labels=[f"D{i}" for i in range(n_tiers, 0, -1)])




Helpers loaded.


## Part A — Load raw data

In [4]:
# ── A1. Shopify order transactions ────────────────────────────────────────────
assert ORDER_FILES, f"No 1_*.xlsx files in {RAW_ORDERS_DIR}"

raw_chunks = []
for f in ORDER_FILES:
    print(f"  {f.name} ...", end=" ")
    df = pd.read_excel(f, dtype={"ID": str, "Customer: ID": str})
    print(f"{len(df):,} rows")
    raw_chunks.append(df)

raw = pd.concat(raw_chunks, ignore_index=True)
raw["Processed At"] = pd.to_datetime(raw["Processed At"], utc=True, errors="coerce")
raw["order_date"] = raw["Processed At"].dt.tz_convert("Asia/Singapore").dt.normalize()
raw["store"] = raw["Name"].apply(store_prefix)

orders_cols = [
    "ID", "Name", "Tags", "order_date", "store", "Customer: ID", "Currency",
    "Price: Total", "Price: Total Discount", "Price: Total Shipping",
    "Payment: Status", "Order Fulfillment Status",
    "Shipping: Country", "Browser: UTM Source", "Browser: UTM Medium",
    "Line: Product Handle", "Line: Title", "Line: Variant Title", "Line: SKU",
    "Line: Price", "Line: Quantity",
]
existing_cols = [c for c in orders_cols if c in raw.columns]
orders_df = raw[raw["Top Row"] == 1][existing_cols].copy()
orders_df = orders_df.rename(columns={"ID": "order_id", "Customer: ID": "customer_id"})
orders_df = orders_df.dropna(subset=["customer_id", "order_date"])
if "Payment: Status" in orders_df.columns:
    orders_df = orders_df[
        orders_df["Payment: Status"].isin(["paid", "partially_refunded"]) | orders_df["Payment: Status"].isna()
    ]
orders_df = orders_df[orders_df["Order Fulfillment Status"].fillna("") != "restocked"]

orders_df["_fx"] = orders_df["store"].map(FX_RATES_TO_SGD).fillna(1.0)
for col in ["Price: Total", "Price: Total Discount", "Price: Total Shipping", "Line: Price"]:
    if col in orders_df.columns:
        orders_df[col] = pd.to_numeric(orders_df[col], errors="coerce").fillna(0) * orders_df["_fx"]
orders_df.drop(columns=["_fx"], inplace=True)
orders_df["Currency"] = "SGD"
orders_df["channel"] = orders_df.apply(classify_channel, axis=1)
orders_df["product_category"] = orders_df["Line: Product Handle"].apply(classify_product)
orders_df["has_discount"] = orders_df["Price: Total Discount"].fillna(0) > 0
orders_df["is_subscription"] = orders_df["Tags"].fillna("").str.lower().str.contains("subscription|yotpo subscriptions")

line_cols = [
    "ID", "Customer: ID", "order_date", "store",
    "Line: Product Handle", "Line: Title", "Line: Variant Title",
    "Line: SKU", "Line: Quantity", "Line: Price", "Line: Discount", "Line: Total",
]
lines_df = raw[raw["Line: Type"] == "Line Item"][[c for c in line_cols if c in raw.columns]].copy()
lines_df = lines_df.rename(columns={"ID": "order_id", "Customer: ID": "customer_id"})
lines_df = lines_df.dropna(subset=["customer_id", "order_date"])
lines_df["product_category"] = lines_df["Line: Product Handle"].apply(classify_product)
lines_df["_fx"] = lines_df["store"].map(FX_RATES_TO_SGD).fillna(1.0)
for col in ["Line: Price", "Line: Discount", "Line: Total"]:
    if col in lines_df.columns:
        lines_df[col] = pd.to_numeric(lines_df[col], errors="coerce").fillna(0) * lines_df["_fx"]
lines_df.drop(columns=["_fx"], inplace=True)

cust_base = (
    orders_df.sort_values("order_date")
    .groupby("customer_id")
    .agg(
        first_order_date=("order_date", "min"),
        last_order_date=("order_date", "max"),
        total_orders=("order_id", "count"),
        total_revenue=("Price: Total", "sum"),
        total_discount=("Price: Total Discount", "sum"),
        ever_subscribed=("is_subscription", "any"),
        ever_discounted=("has_discount", "any"),
        first_channel=("channel", "first"),
        first_product_cat=("product_category", "first"),
        first_store=("store", "first"),
    )
    .reset_index()
)
cust_base["cohort_month"] = cust_base["first_order_date"].dt.to_period("M")
second_orders = (
    orders_df.sort_values("order_date").groupby("customer_id", as_index=False).nth(1)[["customer_id", "order_date"]]
    .rename(columns={"order_date": "second_order_date"})
)
cust_base = cust_base.merge(second_orders, on="customer_id", how="left")
cust_base["days_to_second"] = (cust_base["second_order_date"] - cust_base["first_order_date"]).dt.days
cust_base["is_repeat"] = cust_base["total_orders"] >= 2
cust_base["lifespan_days"] = (cust_base["last_order_date"] - cust_base["first_order_date"]).dt.days
cust_base["recency_days"] = (ANALYSIS_DATE - cust_base["last_order_date"]).dt.days

orders_df = clean_object_cols(orders_df)
lines_df = clean_object_cols(lines_df)
cust_base = clean_object_cols(cust_base)

print(f"\nOrders: {len(orders_df):,} | Lines: {len(lines_df):,} | Customers: {len(cust_base):,}")
print(f"Date range: {orders_df['order_date'].min().date()} → {orders_df['order_date'].max().date()}")

# Aliases for EDA comparisons (raw snapshot before finals filters)
orders_all = orders_df.copy()
lines_all = lines_df.copy()
cust_all = cust_base.copy()




  1_1.orders-2020_20260505.xlsx ... 12,201 rows
  1_2.orders-2021_20260505.xlsx ... 33,196 rows
  1_3.orders-2022_20260505.xlsx ... 18,563 rows
  1_4.orders-2023_20260505.xlsx ... 12,962 rows
  1_5.orders-2024_20260505.xlsx ... 26,373 rows
  1_6.orders-2025_20260505.xlsx ... 40,932 rows
  1_7.orders-2026_20260505.xlsx ... 9,601 rows

Orders: 27,350 | Lines: 50,963 | Customers: 13,780
Date range: 2020-01-01 → 2026-03-31


In [5]:
# ── A2. Ancillary raw tables ──────────────────────────────────────────────────
products_df = pd.read_excel(PRODUCTS_FILE)
discounts_df = pd.read_csv(DISCOUNTS_FILE, encoding="utf-8", encoding_errors="replace")
campaigns_df = pd.read_csv(CAMPAIGNS_FILE, encoding="utf-8", encoding_errors="replace")

rc_orders = pd.read_excel(_glob_one(RECHARGE_DIR, "*orders_combined*.xlsx"))
rc_checkout = pd.read_excel(_glob_one(RECHARGE_DIR, "*checkout*.xlsx"))
rc_reactivated = pd.read_excel(_glob_one(RECHARGE_DIR, "*reactivated*.xlsx"))
rc_churned = pd.read_excel(_glob_one(RECHARGE_DIR, "*churned*.xlsx"))
rc_recurring = pd.read_excel(_glob_one(RECHARGE_DIR, "*recurring*.xlsx"))

cost_map = build_cogs_map(products_df)
print("Products:", len(products_df), "| Discounts:", len(discounts_df), "| Campaigns:", len(campaigns_df))
print("Recharge tables:", len(rc_orders), len(rc_checkout), len(rc_reactivated), len(rc_churned), len(rc_recurring))
print("SKUs with COGS from product master:", len(cost_map))


Products: 167 | Discounts: 367 | Campaigns: 137033
Recharge tables: 1215 1094 50 526 650
SKUs with COGS from product master: 53


## Part B — Build finals cohort (DQ + LP filters)

Mirrors `EDA/13_build_finals_datasets.py`:
- **Layer 1:** DQ-02/03/04 order drops
- **Layer 2:** LP-F01/F02/F03/F04 customer flags
- **Layer 0:** 2022+ order window
- **Layer 3:** Drop Jul/Nov order months + elite SKU lines


In [6]:
def rebuild_customers(orders_df, lines_df, cust_seed, analysis_date=ANALYSIS_DATE):
    orders_df = orders_df.copy()
    orders_df["_rev_sgd"] = pd.to_numeric(orders_df["Price: Total"], errors="coerce").fillna(0)
    agg = (
        orders_df.groupby("customer_id")
        .agg(total_orders=("order_id", "count"), total_revenue=("_rev_sgd", "sum"), last_order_date=("order_date", "max"))
        .reset_index()
    )
    orders_df.drop(columns=["_rev_sgd"], inplace=True, errors="ignore")

    active_ids = set(agg["customer_id"])
    cust = cust_seed[cust_seed["customer_id"].isin(active_ids)].copy()
    drop_cols = [
        "total_orders", "total_revenue", "first_order_date", "last_order_date", "is_repeat",
        "acq_year", "acq_month", "lifespan_days", "recency_days", "days_to_second", "second_order_date",
        "first_disc_depth", "first_disc_bin", "first_order_source", "first_order_pos",
        "exclude_elite_buyer", "exclude_51pct", "exclude_promo_month", "finals_eligible",
    ]
    cust = cust.drop(columns=[c for c in drop_cols if c in cust.columns])
    cust = cust.merge(agg, on="customer_id", how="inner")

    acq_cols = cust_seed[["customer_id", "first_order_date"]].drop_duplicates("customer_id")
    cust = cust.merge(acq_cols, on="customer_id", how="left")

    cust["is_repeat"] = cust["total_orders"] >= 2
    cust["acq_year"] = cust["first_order_date"].dt.year
    cust["acq_month"] = cust["first_order_date"].dt.month
    cust["lifespan_days"] = (cust["last_order_date"] - cust["first_order_date"]).dt.days
    cust["recency_days"] = (analysis_date - cust["last_order_date"]).dt.days

    first_ord = orders_df.sort_values("order_date").groupby("customer_id").first().reset_index()
    first_ord["first_rev"] = pd.to_numeric(first_ord["Price: Total"], errors="coerce").fillna(0)
    first_ord["first_disc"] = pd.to_numeric(first_ord["Price: Total Discount"], errors="coerce").fillna(0)
    first_ord["first_disc_depth"] = np.where(
        (first_ord["first_rev"] + first_ord["first_disc"]) > 0,
        first_ord["first_disc"] / (first_ord["first_rev"] + first_ord["first_disc"]), 0,
    )
    first_ord["first_disc_bin"] = pd.cut(
        first_ord["first_disc_depth"],
        bins=[-0.001, 0.001, 0.05, 0.10, 0.20, 0.30, 0.50, 1.01],
        labels=["0%", "1-5%", "6-10%", "11-20%", "21-30%", "31-50%", "51%+"],
    )
    if "Source" in first_ord.columns:
        first_ord["first_order_source"] = first_ord["Source"].fillna("unknown").str.lower()
    else:
        first_ord["first_order_source"] = "unknown"

    second_ord = orders_df.sort_values("order_date").groupby("customer_id", as_index=False).nth(1)[["customer_id", "order_date"]]
    second_ord = second_ord.rename(columns={"order_date": "second_order_date"})
    cust = cust.merge(first_ord[["customer_id", "first_disc_depth", "first_disc_bin", "first_order_source"]], on="customer_id", how="left")
    cust = cust.merge(second_ord, on="customer_id", how="left")
    cust["days_to_second"] = (cust["second_order_date"] - cust["first_order_date"]).dt.days
    cust["first_order_pos"] = cust["first_order_source"] == "pos"

    for col in ["first_channel", "ever_subscribed", "ever_discounted", "first_product_cat"]:
        if col not in cust.columns and col in cust_seed.columns:
            cust = cust.merge(cust_seed[["customer_id", col]].drop_duplicates("customer_id"), on="customer_id", how="left")

    elite_customers = set(
        lines_df[lines_df["Line: Product Handle"].fillna("").str.contains(EXCLUDE_HANDLE, case=False)]["customer_id"]
    )
    cust["exclude_elite_buyer"] = cust["customer_id"].isin(elite_customers)
    cust["exclude_51pct"] = cust["first_disc_bin"].astype(str) == "51%+"
    cust["exclude_promo_month"] = cust["acq_month"].isin(EXCLUDE_MONTHS)
    cust["finals_eligible"] = (
        (cust["first_order_date"] >= ANALYSIS_START)
        & ~cust["exclude_elite_buyer"]
        & ~cust["exclude_51pct"]
        & ~cust["exclude_promo_month"]
    )
    return cust


# Enrich orders with POS/web Source from raw files
src_chunks = []
for f in ORDER_FILES:
    if f.stat().st_size < 1000:
        continue
    src = pd.read_excel(f, usecols=["ID", "Top Row", "Source"], dtype=str, engine="openpyxl")
    src = src[src["Top Row"] == "1"].rename(columns={"ID": "order_id"})
    src_chunks.append(src[["order_id", "Source"]])
if src_chunks:
    src_df = pd.concat(src_chunks, ignore_index=True).drop_duplicates("order_id")
    src_df["order_id"] = src_df["order_id"].astype(str)
    orders_df["order_id"] = orders_df["order_id"].astype(str)
    orders_df = orders_df.merge(src_df, on="order_id", how="left")

orders_df["order_date"] = pd.to_datetime(orders_df["order_date"], utc=True)
lines_df["order_date"] = pd.to_datetime(lines_df["order_date"], utc=True)
orders_df["order_id"] = orders_df["order_id"].astype(str)
lines_df["order_id"] = lines_df["order_id"].astype(str)
cust_base["first_order_date"] = pd.to_datetime(cust_base["first_order_date"], utc=True)

# Layer 1 — DQ
_rev = pd.to_numeric(orders_df["Price: Total"], errors="coerce").fillna(0)
_disc = pd.to_numeric(orders_df["Price: Total Discount"], errors="coerce").fillna(0)
_dq02 = (_rev == 0) & (_disc == 0)
_dq03 = (_rev == 0) & (_disc > 0)
_dq04 = orders_df["Tags"].fillna("").str.lower().str.contains("wholesale") | (_rev > 5000)
orders_dq = orders_df[~(_dq02 | _dq03 | _dq04)].copy()
lines_dq = lines_df[lines_df["order_id"].isin(set(orders_dq["order_id"]))].copy()
cust_dq = rebuild_customers(orders_dq, lines_dq, cust_base)

# Layer 2
finals_ids = set(cust_dq[cust_dq["finals_eligible"]]["customer_id"])
orders_l2 = orders_dq[orders_dq["customer_id"].isin(finals_ids)].copy()
lines_l2 = lines_dq[lines_dq["customer_id"].isin(finals_ids)].copy()
cust_l2 = cust_dq[cust_dq["finals_eligible"]].copy()

# Layer 0
orders_l2 = orders_l2[orders_l2["order_date"] >= ANALYSIS_START].copy()
lines_l2 = lines_l2[lines_l2["order_id"].isin(set(orders_l2["order_id"]))].copy()

# Layer 3
_jul_nov = orders_l2["order_date"].dt.month.isin(EXCLUDE_MONTHS)
orders_finals = orders_l2[~_jul_nov].copy()
lines_finals = lines_l2[
    lines_l2["order_id"].isin(set(orders_finals["order_id"]))
    & ~lines_l2["Line: Product Handle"].fillna("").str.contains(EXCLUDE_HANDLE, case=False)
].copy()

customers = rebuild_customers(orders_finals, lines_finals, cust_l2)
customers = customers[customers["customer_id"].isin(finals_ids)].copy()
customers["finals_eligible"] = True
orders = orders_finals.copy()
lines = lines_finals.copy()

print("Finals cohort:")
print(f"  Orders:    {len(orders):,}")
print(f"  Lines:     {len(lines):,}")
print(f"  Customers: {len(customers):,}")
print(f"  Finals-eligible flag: {customers['finals_eligible'].sum():,}")


Finals cohort:
  Orders:    8,955
  Lines:     14,448
  Customers: 5,694
  Finals-eligible flag: 5,694


## Part C — Margin enrichment (COGS from product master)

In [7]:
OUT_DIR = STANDALONE_OUT / "eda"
OUT_DIR.mkdir(exist_ok=True)

# Attach unit costs and gross profit to line items
lines["sku_key"] = lines["Line: SKU"].astype(str).str.strip()
lines["unit_cost"] = lines["sku_key"].map(cost_map)
lines["line_rev"] = pd.to_numeric(lines["Line: Total"], errors="coerce").fillna(0)
lines["qty"] = pd.to_numeric(lines["Line: Quantity"], errors="coerce").fillna(1).clip(lower=1)
lines["cogs"] = lines["unit_cost"] * lines["qty"]
lines["gross_profit"] = np.where(lines["unit_cost"].notna(), lines["line_rev"] - lines["cogs"], lines["line_rev"] * MARGIN_PROXY)
lines["has_cogs"] = lines["unit_cost"].notna()
lines["margin_pct"] = np.where(lines["line_rev"] > 0, lines["gross_profit"] / lines["line_rev"], np.nan)

order_gp = lines.groupby("order_id").agg(
    order_gp=("gross_profit", "sum"), order_cogs=("cogs", "sum"), order_rev=("line_rev", "sum"),
    n_categories=("product_category", "nunique"),
).reset_index()
order_gp["order_margin_pct"] = np.where(order_gp["order_rev"] > 0, order_gp["order_gp"] / order_gp["order_rev"], np.nan)
orders = orders.merge(order_gp, on="order_id", how="left")
orders["order_gp"] = orders["order_gp"].fillna(pd.to_numeric(orders["Price: Total"], errors="coerce").fillna(0) * MARGIN_PROXY)

cat_ever = lines.groupby("customer_id")["product_category"].nunique().reset_index(name="n_categories_ever")
cust_rev = orders.groupby("customer_id").agg(
    finals_revenue=("Price: Total", lambda s: pd.to_numeric(s, errors="coerce").sum()),
    finals_orders=("order_id", "count"),
    true_gross_profit=("order_gp", "sum"),
).reset_index()
customers = customers.merge(cat_ever, on="customer_id", how="left").merge(cust_rev, on="customer_id", how="left")
customers["true_gross_profit"] = customers["true_gross_profit"].fillna(customers["total_revenue"] * MARGIN_PROXY)
customers["n_categories_ever"] = customers["n_categories_ever"].fillna(1).astype(int)

covered = lines[lines["has_cogs"]]
cust_gp = covered.groupby("customer_id").agg(
    true_gp_covered=("gross_profit", "sum"), rev_covered=("line_rev", "sum"),
).reset_index()
rev_all = lines.groupby("customer_id")["line_rev"].sum().reset_index(name="rev_total_lines")
cust_gp = rev_all.merge(cust_gp, on="customer_id", how="left")
cust_gp["cogs_coverage_pct"] = np.where(
    cust_gp["rev_total_lines"] > 0, cust_gp["rev_covered"].fillna(0) / cust_gp["rev_total_lines"], 0,
)
customers = customers.merge(cust_gp[["customer_id", "cogs_coverage_pct"]], on="customer_id", how="left")
customers["cogs_coverage_pct"] = customers["cogs_coverage_pct"].fillna(0)
customers["avg_margin_pct"] = np.where(
    customers["finals_revenue"] > 0, customers["true_gross_profit"] / customers["finals_revenue"], np.nan,
)

pool = customers[customers["finals_eligible"]].copy()
pool["profit_decile_true"] = assign_decile(pool["true_gross_profit"])
pool["freq_decile_true"] = assign_decile(pool["finals_orders"].fillna(pool["total_orders"]))
pool["is_top_profit"] = pool["profit_decile_true"] == "D1"
pool["is_top_freq"] = pool["freq_decile_true"] == "D1"
pool["is_top_both"] = pool["is_top_profit"] & pool["is_top_freq"]
pool["crm_tier"] = pool.apply(crm_tier, axis=1)
customers = customers.merge(
    pool[["customer_id", "profit_decile_true", "freq_decile_true", "is_top_profit", "is_top_freq", "is_top_both", "crm_tier"]],
    on="customer_id", how="left",
)

# Save standalone parquet cache (optional — for inspection)
for name, df in [("orders", orders), ("lines", lines), ("customers", customers)]:
    df.to_parquet(OUT_DIR / f"{name}.parquet", index=False)

manifest = {
    "source": "eda_from_raw.ipynb",
    "row_counts": {"orders": len(orders), "lines": len(lines), "customers": len(customers)},
    "pct_lines_with_cogs": round(lines["has_cogs"].mean() * 100, 1),
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))



{
  "source": "solution2_standalone_from_raw.ipynb",
  "row_counts": {
    "orders": 8955,
    "lines": 14448,
    "customers": 5694
  },
  "pct_lines_with_cogs": 21.0
}


## Part D — Rebuild filter funnel with step-by-step counts


In [ ]:
# Recompute funnel on copies for visualization (mirrors 13_build_finals_datasets.py)
o = orders_all.copy()
l = lines_all.copy()
c = cust_all.copy()

if "Payment: Status" in o.columns:
    o = o[o["Payment: Status"].isin(["paid", "partially_refunded"]) | o["Payment: Status"].isna()]
o = o[o["Order Fulfillment Status"].fillna("") != "restocked"]

_rev = pd.to_numeric(o["Price: Total"], errors="coerce").fillna(0)
_disc = pd.to_numeric(o["Price: Total Discount"], errors="coerce").fillna(0)
dq02 = (_rev == 0) & (_disc == 0)
dq03 = (_rev == 0) & (_disc > 0)
dq04 = o["Tags"].fillna("").str.lower().str.contains("wholesale") | (_rev > 5000)
o_dq = o[~(dq02 | dq03 | dq04)].copy()
l_dq = l[l["order_id"].isin(set(o_dq["order_id"]))].copy()
c_dq = rebuild_customers(o_dq, l_dq, c)

finals_ids = set(c_dq[c_dq["finals_eligible"]]["customer_id"])
o_l2 = o_dq[o_dq["customer_id"].isin(finals_ids)].copy()
o_l0 = o_l2[o_l2["order_date"] >= ANALYSIS_START].copy()
_jul = o_l0["order_date"].dt.month.isin(EXCLUDE_MONTHS)
o_f = o_l0[~_jul].copy()

funnel = pd.DataFrame({
    "step": [
        "Raw orders", "After payment/fulfillment", "After DQ-02/03/04",
        "LP-F finals customers", "Orders 2022+", "Excl Jul/Nov months",
    ],
    "orders": [len(orders_all), len(o), len(o_dq), len(o_l2), len(o_l0), len(o_f)],
    "customers": [len(c), c["customer_id"].nunique(), c_dq["customer_id"].nunique(),
                  len(finals_ids), o_l0["customer_id"].nunique(), o_f["customer_id"].nunique()],
})
display(funnel)

dq_counts = pd.Series({"DQ-02 zero/zero": dq02.sum(), "DQ-03 free fulfilment": dq03.sum(), "DQ-04 wholesale/>5k": dq04.sum()})
lp_counts = pd.Series({
    "LP-F03 pre-2022 acq": (~(c_dq["first_order_date"] >= ANALYSIS_START)).sum(),
    "LP-F01 elite SKU": c_dq["exclude_elite_buyer"].sum(),
    "LP-F02 Jul/Nov acq": c_dq["exclude_promo_month"].sum(),
    "LP-F04 51%+ 1st disc": c_dq["exclude_51pct"].sum(),
    "Finals eligible": c_dq["finals_eligible"].sum(),
})
print("\nDQ drops:", dq_counts.to_dict())
print("LP flags:", lp_counts.to_dict())


## Part E — Visualizations: raw data profile


In [ ]:
OUT = STANDALONE_OUT / "eda"
OUT.mkdir(exist_ok=True)

# 1. Orders by year
oy = orders_all.assign(year=orders_all["order_date"].dt.year).groupby("year").size()
fig, ax = plt.subplots(figsize=(8, 4))
oy.plot(kind="bar", ax=ax, color="#457B9D", edgecolor="white")
ax.set_title("Raw Order Volume by Year")
ax.set_ylabel("Orders")
plt.tight_layout()
plt.savefig(OUT / "01_orders_by_year.png", dpi=150)
plt.show()

# 2. Revenue distribution (log)
rev = pd.to_numeric(orders_all["Price: Total"], errors="coerce").fillna(0)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(rev[rev > 0], bins=60, color="#A8DADC", edgecolor="white")
ax.set_xscale("log")
ax.set_title("Order Revenue Distribution (SGD, log scale)")
plt.tight_layout()
plt.savefig(OUT / "02_revenue_distribution.png", dpi=150)
plt.show()

# 3. Channel mix — raw vs finals
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
orders_all["channel"].value_counts().plot(kind="barh", ax=axes[0], color="#E76F51")
axes[0].set_title("Raw — Channel Mix")
orders["channel"].value_counts().plot(kind="barh", ax=axes[1], color="#2E86AB")
axes[1].set_title("Finals Cohort — Channel Mix")
plt.tight_layout()
plt.savefig(OUT / "03_channel_raw_vs_finals.png", dpi=150)
plt.show()

# 4. Store mix
orders_all["store"].value_counts().plot(kind="pie", autopct="%1.1f%%", figsize=(6, 6), ylabel="")
plt.title("Orders by Store (SG/MY/HK)")
plt.savefig(OUT / "04_store_mix.png", dpi=150)
plt.show()

# 5. Product category mix
cat = lines_all.groupby("product_category")["Line: Total"].sum().sort_values(ascending=False).head(8)
fig, ax = plt.subplots(figsize=(8, 4))
cat.plot(kind="bar", ax=ax, color="#F4A261", edgecolor="white")
ax.set_title("Line Revenue by Product Category (raw)")
ax.set_ylabel("SGD")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(OUT / "05_category_revenue.png", dpi=150)
plt.show()


In [ ]:
# 6. Filter funnel waterfall — orders
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(funnel["step"], funnel["orders"], color="#457B9D", edgecolor="white")
ax.set_title("Filter Funnel — Orders Remaining at Each Step")
ax.set_xlabel("Order count")
for i, v in enumerate(funnel["orders"]):
    ax.text(v + 100, i, f"{v:,}", va="center")
plt.tight_layout()
plt.savefig(OUT / "06_funnel_orders.png", dpi=150)
plt.show()

# 7. Filter funnel — customers
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(funnel["step"], funnel["customers"], color="#E76F51", edgecolor="white")
ax.set_title("Filter Funnel — Unique Customers")
ax.set_xlabel("Customer count")
for i, v in enumerate(funnel["customers"]):
    ax.text(v + 50, i, f"{v:,}", va="center")
plt.tight_layout()
plt.savefig(OUT / "07_funnel_customers.png", dpi=150)
plt.show()

# 8. DQ rule breakdown
fig, ax = plt.subplots(figsize=(7, 4))
dq_counts.plot(kind="bar", ax=ax, color=["#E63946", "#F4A261", "#457B9D"], edgecolor="white")
ax.set_title("Data Quality Drops (Layer 1)")
ax.set_ylabel("Orders removed")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(OUT / "08_dq_drops.png", dpi=150)
plt.show()

# 9. LP founder filter flags
fig, ax = plt.subplots(figsize=(8, 4))
lp_counts.plot(kind="bar", ax=ax, color="#2A9D8F", edgecolor="white")
ax.set_title("Founder Filters (LP-F01–F04) — Customer Flags")
ax.set_ylabel("Customers")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(OUT / "09_lp_filters.png", dpi=150)
plt.show()


In [ ]:
# 10. Acquisition cohort heatmap (finals customers)
fc = customers[customers["finals_eligible"] == True].copy()
fc["acq_year"] = fc["first_order_date"].dt.year
fc["acq_month"] = fc["first_order_date"].dt.month
cohort = fc.groupby(["acq_year", "acq_month"]).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(cohort, annot=True, fmt="d", cmap="YlGnBu", ax=ax)
ax.set_title("Finals Cohort — Customer Acquisition Heatmap")
plt.tight_layout()
plt.savefig(OUT / "10_acquisition_heatmap.png", dpi=150)
plt.show()

# 11. Promo month impact (Jul/Nov order share)
om = orders.copy()
om["month"] = om["order_date"].dt.month
month_cnt = om["month"].value_counts().sort_index()
colors_m = ["#E63946" if m in EXCLUDE_MONTHS else "#457B9D" for m in month_cnt.index]
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(month_cnt.index, month_cnt.values, color=colors_m, edgecolor="white")
ax.set_title("Finals Orders by Calendar Month (Jul/Nov excluded in Layer 3)")
ax.set_xlabel("Month")
plt.tight_layout()
plt.savefig(OUT / "11_orders_by_month.png", dpi=150)
plt.show()

# 12. First-order discount depth
fd = customers["first_disc_depth"].dropna()
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(fd, bins=30, color="#A8DADC", edgecolor="white")
ax.axvline(0.51, color="red", ls="--", label="LP-F04 threshold (51%)")
ax.set_title("First Retained Order — Discount Depth")
ax.set_xlabel("Discount depth")
ax.legend()
plt.tight_layout()
plt.savefig(OUT / "12_first_disc_depth.png", dpi=150)
plt.show()


In [ ]:
# 13. Repeat vs one-time (finals)
rep = customers["is_repeat"].value_counts()
fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(rep.values, labels=["One-time", "Repeat"], autopct="%1.1f%%", colors=["#E76F51", "#2E86AB"])
ax.set_title("Finals Customers: Repeat vs One-time")
plt.savefig(OUT / "13_repeat_rate.png", dpi=150)
plt.show()

# 14. COGS coverage
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(customers["cogs_coverage_pct"].fillna(0), bins=20, color="#457B9D", edgecolor="white")
ax.set_title("COGS Coverage % per Customer (lines with real unit cost)")
ax.set_xlabel("Share of line revenue with COGS")
plt.tight_layout()
plt.savefig(OUT / "14_cogs_coverage.png", dpi=150)
plt.show()

# 15. Monthly revenue trend (finals)
mo = orders.copy()
mo["ym"] = mo["order_date"].dt.to_period("M")
monthly_rev = mo.groupby("ym")["Price: Total"].sum()
fig, ax = plt.subplots(figsize=(12, 4))
monthly_rev.plot(ax=ax, color="#2E86AB", lw=2)
ax.set_title("Finals Cohort — Monthly Revenue (SGD)")
ax.set_ylabel("Revenue")
plt.tight_layout()
plt.savefig(OUT / "15_monthly_revenue.png", dpi=150)
plt.show()

# 16. Subscription rate over acquisition year
sub = fc.groupby("acq_year")["ever_subscribed"].mean()
fig, ax = plt.subplots(figsize=(8, 4))
sub.plot(kind="bar", ax=ax, color="#F4A261", edgecolor="white")
ax.set_title("Subscription Rate by Acquisition Year")
ax.set_ylabel("Share ever subscribed")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
plt.tight_layout()
plt.savefig(OUT / "16_sub_rate_by_year.png", dpi=150)
plt.show()

# 17. Before/after comparison panel
compare = pd.DataFrame({
    "metric": ["Orders", "Customers", "Median order value"],
    "Raw": [len(orders_all), len(cust_all), orders_all["Price: Total"].median()],
    "Finals": [len(orders), len(customers), orders["Price: Total"].median()],
})
display(compare)

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(compare))
w = 0.35
ax.bar(x - w/2, compare["Raw"], w, label="Raw", color="#A8DADC")
ax.bar(x + w/2, compare["Finals"], w, label="Finals", color="#2E86AB")
ax.set_xticks(x)
ax.set_xticklabels(compare["metric"])
ax.legend()
ax.set_title("Raw vs Finals Cohort — Key Metrics")
plt.tight_layout()
plt.savefig(OUT / "17_raw_vs_finals.png", dpi=150)
plt.show()

print(f"\nSaved 17 charts to {OUT}")


## Part F — Summary


In [ ]:
print("=" * 65)
print("EDA FROM RAW — SUMMARY")
print("=" * 65)
print(f"Raw orders/customers     : {len(orders_all):,} / {len(cust_all):,}")
print(f"Finals orders/customers  : {len(orders):,} / {len(customers):,}")
print(f"COGS line coverage     : {lines['has_cogs'].mean():.1%}")
print(f"Charts saved to        : {OUT}")
print("=" * 65)
